In [ ]:
import sys
from typing import List, Dict, Iterable, Tuple
import numpy





def Backtrack(match_reward, mismatch_penalty, gap_opening, gap_extension, s, t, n, m):
    NEG_INF = float('-inf')
    M = [[NEG_INF] * (m + 1) for _ in range(n + 1)]
    L = [[NEG_INF] * (m + 1) for _ in range(n + 1)]
    U = [[NEG_INF] * (m + 1) for _ in range(n + 1)]

    M[0][0] = 0
    for i in range(1, n + 1):
        L[i][0] = -gap_opening - (i - 1) * gap_extension
        M[i][0] = L[i][0]
    for j in range(1, m + 1):
        U[0][j] = -gap_opening - (j - 1) * gap_extension
        M[0][j] = U[0][j]

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            L[i][j] = max(L[i-1][j] - gap_extension,
                          M[i-1][j] - gap_opening)
            U[i][j] = max(U[i][j-1] - gap_extension,
                          M[i][j-1] - gap_opening)
            sub = match_reward if s[i-1] == t[j-1] else -mismatch_penalty
            M[i][j] = max(M[i-1][j-1] + sub, L[i][j], U[i][j])
    # print(M,L,U)
    return M, L, U


def OutputAlignment(M, L, U, match_reward, mismatch_penalty, gap_opening, gap_extension, s, t, i, j):
    aligned_s, aligned_t = [], []
    layer = 'M'
    if L[i][j] > M[i][j]:
        layer = 'L'
    if U[i][j] > max(M[i][j], L[i][j]):
        layer = 'U'

    while i > 0 or j > 0:
        layer, i, j, s1, t1 = TakeStep(M, L, U, match_reward, mismatch_penalty, gap_opening, gap_extension, s, t, i, j, layer)


        if s1 != '': 
            aligned_s.append(s1)
            aligned_t.append(t1)

    return ''.join(reversed(aligned_s)), ''.join(reversed(aligned_t))

def TakeStep(M, L, U, match_reward, mismatch_penalty, gap_opening, gap_extension, s, t, i, j, layer):
    if layer == 'L':
        if i > 0 and L[i][j] == M[i-1][j] - gap_opening:
            return 'M', i-1, j, s[i-1], '-'
        else:
            return 'L', i-1, j, s[i-1], '-'
    elif layer == 'U':
        if j > 0 and U[i][j] == M[i][j-1] - gap_opening:
            return 'M', i, j-1, '-', t[j-1]
        else:
            return 'U', i, j-1, '-', t[j-1]
    else:
        if i > 0 and M[i][j] == L[i][j]:
            return 'L', i, j, '', ''
        elif j > 0 and M[i][j] == U[i][j]:
            return 'U', i, j, '', ''
        else:
            sub = match_reward if s[i-1] == t[j-1] else -mismatch_penalty
            return 'M', i-1, j-1, s[i-1], t[j-1]
        
def AffineAlignment(match_reward: int, mismatch_penalty: int,
                    gap_opening_penalty: int, gap_extension_penalty: int,
                    s: str, t: str) -> Tuple[int, str, str]:
    n, m = len(s), len(t)
    M, L, U = Backtrack(match_reward, mismatch_penalty, gap_opening_penalty, gap_extension_penalty, s, t, n, m)

    aligned_s, aligned_t = OutputAlignment(M, L,U, match_reward, mismatch_penalty, gap_opening_penalty, gap_extension_penalty, s, t, n, m)
    return M[n][m], aligned_s, aligned_t


In [ ]:

# Code Challenge: Solve the Alignment with Affine Gap Penalties Problem.

# Input: A match reward, a mismatch penalty, a gap opening penalty, a gap extension penalty, and two nucleotide strings.
# Output: The maximum alignment score between v and w, followed by an alignment of v and w achieving this maximum score.
# Debug Datasets

# Sample Input 1:

# 1 3 2 1
# GA
# GTTA
# Sample Output 1:

# -1
# G--A
# GTTA
# Sample Input 2:

# 1 5 3 1
# TTT
# TT
# Sample Output 2:

# -1
# TTT
# TT-
# Sample Input 3:

# 1 5 5 1
# GAT
# AT
# Sample Output 3:

# -3
# GAT
# -AT
# Sample Input 4:

# 1 5 2 1
# CCAT
# GAT
# Sample Output 4:

# -3
# CC-AT
# --GAT
# Sample Input 5:

# 1 2 3 2
# CAGGT
# TAC
# Sample Output 5:

# -8
# CAGGT
# TAC--
# Sample Input 6:

# 2 3 3 2
# GTTCCAGGTA
# CAGTAGTCGT
# Sample Output 6:

# -8
# --GTTCCAG--GTA
# CAGT---AGTCGT-
# Sample Input 7:

# 1 3 1 1
# AGCTAGCCTAG
# GT
# Sample Output 7:

# -7
# AGCTAGCCTAG
# -G-T-------
# Sample Input 8:

# 2 1 2 1
# AA
# CAGTGTCAGTA
# Sample Output 8:

# -7
# -A--------A
# CAGTGTCAGTA
# Sample Input 9:

# 5 2 15 5
# ACGTA
# ACT
# Sample Output 9:

# -12
# ACGTA
# ACT--